# 08 · Canciones nuevas: ¿entraría en la playlist?

Proyecto «Playlist perfecta para viajes en grupo» · NPD Entertainment · CEBANC · Ander Ameztoy

El recomendador de los notebooks 06 y 07 ordena las canciones **que el grupo ya conoce**. Ante una canción que nadie ha escuchado, la regla actual casi nunca la deja entrar:

- su gusto siempre es una predicción, y compite con canciones cuyo gusto es la tasa real (muchas 0,9 o 1);
- el modelo GB absoluto predice 1 a casi todas las canciones de Daniel, así que con él una canción nueva empata con cientos y no pasa del percentil 0,5. El gusto del grupo es el mínimo, y el puesto 50 suele estar por encima de 0,75.

Simulando una canción nueva de cada artista del catálogo, en días laborables no entra **ninguna**.

Este notebook plantea un **modelo de novedades** aparte:

1. **Unidad:** la primera vez que una persona escucha una canción.
2. **Qué se predice:** si la **adopta**, es decir, si vuelve a escucharla sin saltarla en los 90 días siguientes. Se usa esto y no «no saltar», porque Daniel casi no salta nada (0,95 también en canciones nuevas) y saltar no distingue sus gustos.
3. **Variables**, todas calculadas solo con escuchas **anteriores** a ese momento:
   - lo abierta que es la persona a lo nuevo;
   - cuánto conoce al artista y al álbum;
   - cuántas canciones nuevas de ese artista acabó adoptando;
   - si el resto del grupo conoce al artista o a la canción;
   - el contexto y si sonó en shuffle.
4. **En la playlist:** las novedades no compiten con las canciones conocidas, sino que tienen sus propios **huecos**. Una canción nueva **entra** si está entre el 20 % de novedades más prometedoras **para cada miembro**, con la misma lógica de mínimo que el recomendador.

**Reglas para no hacer trampa con el lote 2** (las mismas del notebook 07):
- Todas las decisiones se toman con una validación temporal: entrenar con primeras escuchas hasta octubre de 2024 y validar con las de enero a marzo de 2025.
- El lote 2 (desde julio de 2025) se usa **una sola vez**, al final.
- Ninguna celda muestra nombres de canciones ni de artistas: los historiales son personales.

## 0. Configuración

In [ ]:
from pathlib import Path
import os, re, json, glob, warnings, platform, unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss
from sklearn.inspection import permutation_importance
import joblib
warnings.filterwarnings("ignore", category=UserWarning)

try:
    from google.colab import drive
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    drive.mount("/content/drive")
    RAIZ = Path("/content/drive/MyDrive/proyecto-Ebaluacion1")
else:
    RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

CARPETA_HISTORIAL = RAIZ / "data" / "raw"
if not (CARPETA_HISTORIAL / "ander").exists():
    CARPETA_HISTORIAL = RAIZ / "data" / "entrada" / "historial"
HISTORIALES = {
    "ander":  CARPETA_HISTORIAL / "ander",
    "daniel": CARPETA_HISTORIAL / "daniel",
}
CAMPOS = ["ts", "ms_played", "spotify_track_uri", "master_metadata_track_name", "master_metadata_album_artist_name",
          "master_metadata_album_album_name", "reason_end", "shuffle", "platform"]
ZONA = "Europe/Madrid"
CORTE = pd.Timestamp("2025-07-01", tz=ZONA)          # lote 1 < corte <= lote 2
INICIO_VAL = pd.Timestamp("2025-01-01", tz=ZONA)     # validación: primeras escuchas de 2025-01 a 2025-03
UMBRAL_SALTO_MS, SEMILLA = 30_000, 42
VENTANA = pd.Timedelta(days=90)                      # «adopta» = la vuelve a escuchar sin saltarla en 90 días
K = 5                                                # suavizado de las tasas (como en los notebooks 05 a 07)
DIAS_APERTURA = 180                                  # ventana de las variables de apertura de la persona
N_LISTA, UMBRAL_PCT = 50, 0.80                       # playlist de 50; entra si está en el 20 % mejor de cada miembro

COLOR = {"ander": "#56BE91", "daniel": "#FD3B1F"}
NEGRO, GRIS = "#020202", "#B9B6AE"
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "axes.titlelocation": "left"})
FIGURAS = RAIZ / "notebooks" / "figuras"; FIGURAS.mkdir(parents=True, exist_ok=True)
SALIDA = RAIZ / "data" / "salida"; SALIDA.mkdir(parents=True, exist_ok=True)
def guardar(fig, nombre):
    ruta = FIGURAS / f"{nombre}.png"; fig.savefig(ruta, dpi=200, bbox_inches="tight"); return ruta
print("Entorno:", "Google Colab" if EN_COLAB else "local", "| raíz:", RAIZ)

### MLflow

Servidor local de MLflow si está en marcha; si no, la base SQLite `mlflow.db` (en Colab, la copia de Drive).

In [ ]:
try:
    import mlflow
except ImportError:
    if EN_COLAB:
        os.system("pip install -q mlflow"); import mlflow
    else:
        mlflow = None; print("AVISO: mlflow no está instalado; los resultados se calculan igual pero no se registran.")

if mlflow is not None:
    uri = os.environ.get("MLFLOW_TRACKING_URI")
    if uri is None:
        try:
            import urllib.request
            urllib.request.urlopen("http://localhost:5000", timeout=2); uri = "http://localhost:5000"
        except Exception:
            carpeta_mlflow = Path("/content/mlflow") if EN_COLAB else RAIZ / "mlflow"
            carpeta_mlflow.mkdir(parents=True, exist_ok=True)
            previa = RAIZ / "mlflow" / "mlflow.db"
            if EN_COLAB and previa.exists() and not (carpeta_mlflow / "mlflow.db").exists():
                import shutil; shutil.copy2(previa, carpeta_mlflow / "mlflow.db")
                print("Partimos de los experimentos guardados en", previa)
            uri = f"sqlite:///{(carpeta_mlflow / 'mlflow.db').as_posix()}"
    mlflow.set_tracking_uri(uri); mlflow.set_experiment("novedades")
    print("MLflow en:", uri)

def registrar(nombre, params, metricas, figuras=()):
    if mlflow is None: return
    with mlflow.start_run(run_name=nombre):
        mlflow.log_params(params)
        mlflow.log_metrics({k: float(v) for k, v in metricas.items() if pd.notna(v)})
        for f in figuras: mlflow.log_artifact(str(f))

## 1. Datos

Mismas escuchas que en los notebooks 04 a 07. Para cada persona se marca la **primera escucha** de cada canción y si la **adoptó**.

Solo se usan como ejemplos las primeras escuchas con 90 días de margen hasta el final de los datos («evaluables»); si no, no se sabe aún si se adoptaron.

In [ ]:
def leer(carpeta, persona):
    ficheros = sorted(glob.glob(str(carpeta / "**" / "Streaming_History_Audio_*.json"), recursive=True))
    filas = []
    for f in ficheros:
        with open(f, encoding="utf-8") as fh:
            filas += [{k: r.get(k) for k in CAMPOS} for r in json.load(fh)]
    d = pd.DataFrame(filas, columns=CAMPOS); d["persona"] = persona
    return d

h = pd.concat([leer(c, p) for p, c in HISTORIALES.items()], ignore_index=True)
h["ms_played"] = pd.to_numeric(h["ms_played"])
h = h[h["spotify_track_uri"].notna() & (h["ms_played"] > 0)].copy()
h["track_id"] = h["spotify_track_uri"].str.replace("spotify:track:", "", regex=False)
h = h.sort_values("ms_played", ascending=False).drop_duplicates(["persona", "ts", "track_id"]).copy()
h["artista"] = h["master_metadata_album_artist_name"].fillna("?")
h["album"] = h["master_metadata_album_album_name"].fillna("?")
h["hora_local"] = pd.to_datetime(h["ts"], utc=True).dt.tz_convert(ZONA)
h["franja"] = h["hora_local"].dt.hour // 4                                   # 0-5, como en silver
h["tipo_dia"] = np.where(h["hora_local"].dt.dayofweek >= 5, "fin_semana", "laborable")
h["no_salto"] = ~((h["reason_end"] == "fwdbtn") & (h["ms_played"] < UMBRAL_SALTO_MS))
h["shuffle"] = h["shuffle"].astype(bool)
h = h.sort_values("hora_local", kind="stable").reset_index(drop=True)
h["primera"] = ~h.duplicated(["persona", "track_id"])
FIN_DATOS = h["hora_local"].max()

vuelve = h[~h["primera"] & h["no_salto"]].groupby(["persona", "track_id"])["hora_local"].min().rename("t_vuelve")
nuevas = h[h["primera"]].join(vuelve, on=["persona", "track_id"])
nuevas["adopta"] = ((nuevas["t_vuelve"] - nuevas["hora_local"]) <= VENTANA).astype(int)
nuevas["cierre"] = nuevas["hora_local"] + VENTANA                           # desde aquí ya se sabe si la adoptó
nuevas["evaluable"] = nuevas["cierre"] <= FIN_DATOS
nuevas["lote"] = np.where(nuevas["hora_local"] < CORTE, 1, 2)

ev = nuevas[nuevas["evaluable"]]
resumen = pd.DataFrame({
    "escuchas": h.groupby("persona").size(),
    "% escuchas que son canción nueva": h.groupby("persona")["primera"].mean() * 100,
    "canciones nuevas evaluables": ev.groupby("persona").size(),
    "adopta (%)": ev.groupby("persona")["adopta"].mean() * 100,
    "no salta la 1.ª vez (%)": ev.groupby("persona")["no_salto"].mean() * 100,
    "1.ª vez en shuffle (%)": ev.groupby("persona")["shuffle"].mean() * 100,
})
print("Datos hasta", FIN_DATOS.date(), "· primeras escuchas evaluables hasta", (FIN_DATOS - VENTANA).date())
resumen.round(1)

## 2. Variables de cada primera escucha

`variables(consultas)` calcula, para cada consulta (persona, canción, artista, álbum, momento), lo que se sabía **justo antes de ese momento**. Usa `merge_asof` con `allow_exact_matches=False`, así que la propia escucha nunca entra en sus variables.

Esa misma función sirve para entrenar y para preguntar por una canción que nadie ha escuchado.

| Grupo | Variables |
|---|---|
| Apertura de la persona (últimos 180 días) | % de escuchas que son canción nueva, % de no salto en canciones nuevas, % de adopción de sus novedades ya cerradas, tasa de no salto general, % de shuffle |
| Artista | si lo conoce, escuchas previas, tasa de no salto suavizada, canciones distintas suyas que conoce, días desde la última vez, novedades suyas adoptadas y tasa de adopción suavizada |
| Álbum | escuchas previas del álbum |
| Grupo | escuchas y tasa del artista por los **demás** miembros, si alguien más conoce ya la canción |
| Contexto | franja, fin de semana, shuffle |

La adopción de una novedad solo se cuenta cuando han pasado sus 90 días (`cierre`), para no usar información del futuro.

In [ ]:
def _rodante(d, col, dias=DIAS_APERTURA):
    return d.set_index(d.columns[0])[col].astype(float).rolling(f"{dias}D").mean().to_numpy()

# Tablas acumuladas (cada fila: lo que se sabía después de esa escucha)
per = h[["hora_local", "persona"]].copy()
for p, idx in h.groupby("persona").groups.items():
    x = h.loc[idx, ["hora_local", "primera", "no_salto", "shuffle"]]
    per.loc[idx, "per_pct_nuevas"] = _rodante(x, "primera")
    per.loc[idx, "per_tasa"] = _rodante(x, "no_salto")
    per.loc[idx, "per_pct_shuffle"] = _rodante(x, "shuffle")
per_nuevas = nuevas[["hora_local", "persona", "no_salto"]].copy()
per_adop = nuevas.sort_values("cierre", kind="stable")[["cierre", "persona", "adopta"]].copy()
for p in HISTORIALES:
    m = per_nuevas["persona"] == p; per_nuevas.loc[m, "per_ns_nuevas"] = _rodante(per_nuevas[m], "no_salto")
    m = per_adop["persona"] == p; per_adop.loc[m, "per_adopcion"] = _rodante(per_adop[m], "adopta")

con_artista = h[h["artista"] != "?"]
pa = con_artista[["hora_local", "persona", "artista"]].copy()
g = con_artista.groupby(["persona", "artista"])
pa["art_n"] = g.cumcount() + 1
pa["art_ns"] = g["no_salto"].cumsum()
pa["art_canciones"] = g["primera"].cumsum()
pa["art_ultima"] = pa["hora_local"]
alb = con_artista[["hora_local", "persona", "artista", "album"]].copy()
alb["alb_n"] = con_artista.groupby(["persona", "artista", "album"]).cumcount() + 1
tot = con_artista[["hora_local", "artista"]].copy()
tot["tot_n"] = con_artista.groupby("artista").cumcount() + 1
tot["tot_ns"] = con_artista.groupby("artista")["no_salto"].cumsum()
trk = h[["hora_local", "track_id"]].copy()
trk["trk_n"] = h.groupby("track_id").cumcount() + 1
art_adop = nuevas[nuevas["artista"] != "?"].sort_values("cierre", kind="stable")[["cierre", "persona", "artista", "adopta"]].copy()
ga = art_adop.groupby(["persona", "artista"])
art_adop["art_cerradas"] = ga.cumcount() + 1
art_adop["art_adoptadas"] = ga["adopta"].cumsum()

def _asof(izq, der, cuando, por, columnas):
    der = der[[cuando] + por + columnas].rename(columns={cuando: "_t"}).sort_values("_t", kind="stable")
    # merge_asof exige la misma unidad en las dos fechas (pandas 3 lee las fechas en microsegundos)
    der["_t"] = der["_t"].dt.as_unit("ns")
    r = pd.merge_asof(izq, der, left_on="hora_local", right_on="_t", by=por, allow_exact_matches=False)
    return r.drop(columns="_t")

def variables(consultas):
    """Variables de cada consulta (persona, track_id, artista, album, hora_local) con lo escuchado antes de hora_local."""
    f = consultas.reset_index(drop=True).copy()
    f["_id"] = np.arange(len(f))
    f["hora_local"] = pd.to_datetime(f["hora_local"]).dt.tz_convert(ZONA).dt.as_unit("ns")   # misma precisión que h
    f = f.sort_values("hora_local", kind="stable")
    f = _asof(f, per, "hora_local", ["persona"], ["per_pct_nuevas", "per_tasa", "per_pct_shuffle"])
    f = _asof(f, per_nuevas, "hora_local", ["persona"], ["per_ns_nuevas"])
    f = _asof(f, per_adop, "cierre", ["persona"], ["per_adopcion"])
    f = _asof(f, pa, "hora_local", ["persona", "artista"], ["art_n", "art_ns", "art_canciones", "art_ultima"])
    f = _asof(f, alb, "hora_local", ["persona", "artista", "album"], ["alb_n"])
    f = _asof(f, tot, "hora_local", ["artista"], ["tot_n", "tot_ns"])
    f = _asof(f, trk, "hora_local", ["track_id"], ["trk_n"])
    f = _asof(f, art_adop, "cierre", ["persona", "artista"], ["art_cerradas", "art_adoptadas"])
    f = f.sort_values("_id").drop(columns="_id").reset_index(drop=True)
    for c in ["art_n", "art_ns", "art_canciones", "alb_n", "tot_n", "tot_ns", "trk_n", "art_cerradas", "art_adoptadas"]:
        f[c] = f[c].fillna(0)
    return f

def matriz(f):
    """Matriz del modelo. franja, tipo_dia y shuffle son los de la escucha (o los del viaje, si es una consulta)."""
    tasa = f["per_tasa"].fillna(0.5)
    adop = f["per_adopcion"].fillna(0.5)
    grp_n, grp_ns = (f["tot_n"] - f["art_n"]).clip(lower=0), (f["tot_ns"] - f["art_ns"]).clip(lower=0)
    dias = (f["hora_local"] - pd.to_datetime(f["art_ultima"])).dt.total_seconds() / 86400
    return pd.DataFrame({
        "per_pct_nuevas": f["per_pct_nuevas"], "per_ns_nuevas": f["per_ns_nuevas"], "per_adopcion": f["per_adopcion"],
        "per_tasa": f["per_tasa"], "per_pct_shuffle": f["per_pct_shuffle"],
        "art_conocido": (f["art_n"] > 0).astype(int), "art_log_n": np.log1p(f["art_n"]),
        "art_tasa": (f["art_ns"] + K * tasa) / (f["art_n"] + K),
        "art_log_canciones": np.log1p(f["art_canciones"]), "art_log_dias": np.log1p(dias.fillna(3650)),
        "art_log_adoptadas": np.log1p(f["art_adoptadas"]),
        "art_adopcion": (f["art_adoptadas"] + K * adop) / (f["art_cerradas"] + K),
        "alb_log_n": np.log1p(f["alb_n"]),
        "grp_art_log_n": np.log1p(grp_n), "grp_art_tasa": (grp_ns + K * tasa) / (grp_n + K),
        "grp_cancion_conocida": (f["trk_n"] > 0).astype(int),
        "franja": f["franja"].astype(int), "finde": (f["tipo_dia"] == "fin_semana").astype(int),
        "shuffle": f["shuffle"].astype(int),
    })

COLS_CONSULTA = ["persona", "track_id", "artista", "album", "hora_local", "franja", "tipo_dia", "shuffle"]
fn = variables(nuevas[COLS_CONSULTA])
fn[["adopta", "no_salto", "evaluable", "lote", "cierre"]] = nuevas[["adopta", "no_salto", "evaluable", "lote", "cierre"]].to_numpy()
fn["adopta"] = fn["adopta"].astype(int); fn["evaluable"] = fn["evaluable"].astype(bool)
X_todo = matriz(fn)
print(len(fn), "primeras escuchas ·", X_todo.shape[1], "variables")
X_todo.describe().T[["mean", "50%", "min", "max"]].round(3)

### 2.1 ¿Sirven las ideas? Lo abierta que es cada persona y lo que pesa conocer al artista

Antes de entrenar nada se comprueban con los datos del lote 1 las dos ideas de partida:
- que se puede medir lo abierto que es cada uno a lo nuevo;
- que conocer al artista sube la probabilidad de adoptar la canción.

In [ ]:
l1 = fn[fn["evaluable"] & (fn["lote"] == 1)].copy()
l1["artista"] = pd.cut(l1["art_n"], [-1, 0, 4, 19, 99, np.inf], labels=["nuevo", "1-4", "5-19", "20-99", "100+"])
por_artista = l1.pivot_table(index="artista", columns="persona", values="adopta", aggfunc="mean", observed=True) * 100
por_shuffle = l1.pivot_table(index="shuffle", columns="persona", values=["no_salto", "adopta"], aggfunc="mean") * 100
anio = h.assign(anio=h["hora_local"].dt.year).pivot_table(index="anio", columns="persona", values="primera", aggfunc="mean") * 100

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for p in HISTORIALES:
    axes[0].plot(anio.index, anio[p], marker="o", color=COLOR[p], label=p)
    axes[1].plot(por_artista.index.astype(str), por_artista[p], marker="o", color=COLOR[p], label=p)
axes[0].set_title("Apertura: % de escuchas nuevas por año"); axes[0].set_ylabel("%")
axes[1].set_title("Adopción según escuchas previas del artista (lote 1)"); axes[1].set_ylabel("% que adopta")
axes[1].set_xlabel("escuchas previas del artista"); axes[0].legend(frameon=False)
ruta_ideas = guardar(fig, "08_apertura_y_artista"); plt.show()
print("Adopción (%) según el artista:"); display(por_artista.round(1))
print("Primera escucha con y sin shuffle (%):"); display(por_shuffle.round(1))

## 3. Validación temporal (primeras escuchas de enero a marzo de 2025)

- **Entrenamiento:** primeras escuchas cuya ventana de 90 días se cierra antes del 1 de enero de 2025.
- **Validación:** primeras escuchas del 1 de enero al 2 de abril de 2025, que se cierran antes del corte del lote 2.

**Referencias** sin entrenar nada:
- adopción reciente de la persona (`per_adopcion`): solo sabe quién es y lo abierta que está;
- adopción suavizada de sus novedades de ese artista (`art_adopcion`): solo sabe si le gustaron otras canciones nuevas del artista.

**Candidatos:** regresión logística y GB de clasificación, cada uno con todo el historial o con una ventana de 18 meses, la del notebook 05. Se elige por **AUC** de validación; la pérdida logarítmica y Brier se muestran como apoyo.

In [ ]:
def modelos():
    return {
        "logística": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=2000)),
        "GB": HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300, max_leaf_nodes=15, min_samples_leaf=50,
                                             l2_regularization=1.0, random_state=SEMILLA),
    }

def metricas(y, prob, persona):
    prob = np.clip(np.asarray(prob, dtype=float), 1e-4, 1 - 1e-4)
    out = {"AUC": roc_auc_score(y, prob), "log loss": log_loss(y, prob), "Brier": brier_score_loss(y, prob)}
    for p in HISTORIALES:
        m = np.asarray(persona == p)
        out[f"AUC {p}"] = roc_auc_score(y[m], prob[m]) if len(np.unique(y[m])) > 1 else np.nan
    return out

def filas(desde, hasta):
    m = fn["evaluable"] & (fn["hora_local"] < hasta)
    if desde is not None: m &= fn["hora_local"] >= desde
    return fn[m]

FIN_ENT_VAL = INICIO_VAL - VENTANA
ent_todo = filas(None, FIN_ENT_VAL); val = filas(INICIO_VAL, CORTE - VENTANA)
ent_18 = filas(FIN_ENT_VAL - pd.DateOffset(months=18), FIN_ENT_VAL)
X_val, y_val = matriz(val), val["adopta"].to_numpy()
print(f"Entrenamiento: {len(ent_todo)} (todo) / {len(ent_18)} (18 meses) · validación: {len(val)}")

filas_val = {"referencia: adopción de la persona": metricas(y_val, X_val["per_adopcion"].fillna(0.5), val["persona"]),
             "referencia: adopción con el artista": metricas(y_val, X_val["art_adopcion"], val["persona"])}
ajustados = {}
for ventana, ent in [("todo", ent_todo), ("18 meses", ent_18)]:
    for nombre, mod in modelos().items():
        mod.fit(matriz(ent), ent["adopta"])
        ajustados[f"{nombre} · {ventana}"] = mod
        filas_val[f"{nombre} · {ventana}"] = metricas(y_val, mod.predict_proba(X_val)[:, 1], val["persona"])
validacion = pd.DataFrame(filas_val).T
candidatos = validacion.drop(index=[i for i in validacion.index if i.startswith("referencia")])
elegido = candidatos["AUC"].idxmax()
nombre_el, ventana_el = elegido.split(" · ")
print("Elegido:", elegido)
validacion.round(4)

### 3.1 Qué variables pesan (validación, modelo elegido)

Importancia por permutación: cuánto baja el AUC de validación al desordenar cada variable.

In [ ]:
imp = permutation_importance(ajustados[elegido], X_val, y_val, scoring="roc_auc", n_repeats=5, random_state=SEMILLA)
importancia = pd.Series(imp.importances_mean, index=X_val.columns).sort_values()
fig, ax = plt.subplots(figsize=(7, 4.6))
ax.barh(importancia.index, importancia.to_numpy(), color=[COLOR["ander"] if v > 0 else GRIS for v in importancia])
ax.set_title(f"Importancia por permutación · {elegido}"); ax.set_xlabel("caída del AUC de validación")
ruta_imp = guardar(fig, "08_importancia"); plt.show()
importancia.sort_values(ascending=False).head(8).round(4)

## 4. Modelo final y prueba única con el lote 2

- **Modelo:** el elegido, reentrenado con las primeras escuchas cuya ventana se cierra antes del corte, con la misma ventana de historial que se eligió.
- **Prueba:** todas las primeras escuchas del lote 2 con 90 días de margen.

In [ ]:
FIN_ENT = CORTE - VENTANA
ent_fin = filas(None if ventana_el == "todo" else FIN_ENT - pd.DateOffset(months=18), FIN_ENT)
modelo = modelos()[nombre_el].fit(matriz(ent_fin), ent_fin["adopta"])
prueba = fn[fn["evaluable"] & (fn["lote"] == 2)].reset_index(drop=True)
X_p, y_p = matriz(prueba), prueba["adopta"].to_numpy()
prob_p = modelo.predict_proba(X_p)[:, 1]

lote2 = pd.DataFrame({
    "referencia: adopción de la persona": metricas(y_p, X_p["per_adopcion"].fillna(0.5), prueba["persona"]),
    "referencia: adopción con el artista": metricas(y_p, X_p["art_adopcion"], prueba["persona"]),
    f"modelo ({elegido})": metricas(y_p, prob_p, prueba["persona"]),
}).T
nuevo_artista = (X_p["art_conocido"] == 0).to_numpy()
lote2["AUC artista nuevo"] = [roc_auc_score(y_p[nuevo_artista], s[nuevo_artista]) for s in
                              (X_p["per_adopcion"].fillna(0.5), X_p["art_adopcion"], prob_p)]

# Utilidad práctica: de las novedades que el modelo pone en el 20 % mejor de cada persona, ¿cuántas adoptó?
prueba["prob"] = prob_p
prueba["arriba"] = prueba.groupby("persona")["prob"].rank(pct=True) >= UMBRAL_PCT
util = prueba.groupby(["persona", "arriba"])["adopta"].agg(["size", "mean"]).unstack("arriba")
util.columns = [f"{'n' if a == 'size' else 'adopta'} · {'20 % mejor' if b else 'resto'}" for a, b in util.columns]
util[[c for c in util.columns if c.startswith("adopta")]] *= 100
print(f"Entrenamiento final: {len(ent_fin)} filas · prueba: {len(prueba)} primeras escuchas del lote 2")
display(lote2.round(4)); util.round(1)

In [ ]:
prueba["decil"] = pd.qcut(prueba["prob"], 10, labels=False, duplicates="drop")
cal = prueba.groupby("decil").agg(prevista=("prob", "mean"), real=("adopta", "mean"))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].plot([0, 1], [0, 1], color=GRIS, lw=1)
axes[0].plot(cal["prevista"], cal["real"], marker="o", color=NEGRO)
axes[0].set_title("Calibración en el lote 2 (deciles)"); axes[0].set_xlabel("probabilidad prevista"); axes[0].set_ylabel("adopción real")
x = np.arange(len(HISTORIALES))
for i, (col, etiqueta) in enumerate([("adopta · resto", "resto"), ("adopta · 20 % mejor", "20 % mejor según el modelo")]):
    axes[1].bar(x + (i - 0.5) * 0.38, util[col].to_numpy(), 0.38, label=etiqueta,
                color=[GRIS if i == 0 else COLOR[p] for p in util.index])
axes[1].set_xticks(x, util.index); axes[1].set_ylabel("% que adopta"); axes[1].legend(frameon=False)
axes[1].set_title("Lote 2: adopción de las novedades")
ruta_l2 = guardar(fig, "08_lote2"); plt.show()

## 5. «¿Entraría?» y huecos para novedades en la playlist

**Regla**, la misma lógica que el recomendador:
1. Para cada miembro, probabilidad de que adopte la canción. Se calcula en el momento del viaje, con shuffle y como media de los 12 contextos si no se indica horario.
2. Esa probabilidad se convierte en **percentil** frente a las novedades que esa persona escuchó en los 12 meses anteriores. Así se compara a cada uno con lo que suele recibir, y Daniel, que adopta más, no domina.
3. Gusto del grupo = **mínimo** de los percentiles.
4. La canción **entra** si ese mínimo es al menos 0,80, es decir, si está en el 20 % de novedades más prometedoras para todos.
5. **Huecos:** las que entran ocupan, como mucho, `round(50 × apertura del grupo)` puestos (entre 2 y 10), donde la apertura del grupo es la del miembro menos abierto (% de escuchas que son canción nueva en los últimos 180 días). El resto de la lista sigue siendo la del notebook 06.

**Prueba con el lote 2.** Se pregunta, a fecha del corte, por todas las canciones que ninguno de los dos había escuchado antes y que alguno escuchó en el lote 2. Después se mira si de verdad las adoptaron.

In [ ]:
CONTEXTOS = [(fr, td) for fr in range(6) for td in ("laborable", "fin_semana")]

def probabilidades(consultas, franja=None, tipo_dia=None, shuffle=True):
    """Probabilidad de adopción de cada consulta; sin horario, media de los 12 contextos."""
    f = variables(consultas.assign(franja=0, tipo_dia="laborable", shuffle=shuffle))
    ctx = CONTEXTOS if franja is None else [(franja, tipo_dia)]
    return np.mean([modelo.predict_proba(matriz(f.assign(franja=fr, tipo_dia=td)))[:, 1] for fr, td in ctx], axis=0)

def referencia(persona, momento):
    """Probabilidades de las novedades reales de la persona en los 12 meses anteriores (con las que se compara)."""
    r = nuevas[(nuevas["persona"] == persona) & (nuevas["hora_local"] < momento) &
               (nuevas["hora_local"] >= momento - pd.DateOffset(months=12))]
    return np.sort(modelo.predict_proba(matriz(variables(r[COLS_CONSULTA])))[:, 1])

def apertura(persona, momento):
    v = per[(per["persona"] == persona) & (per["hora_local"] < momento)]["per_pct_nuevas"]
    return float(v.iloc[-1]) if len(v) else np.nan

def entraria(personas, canciones, momento, franja=None, tipo_dia=None, refs=None):
    """canciones: DataFrame con track_id, artista, album (vale una canción que nadie ha escuchado).
    Devuelve el percentil de cada miembro, el del grupo (mínimo) y si entra."""
    refs = refs or {p: referencia(p, momento) for p in personas}
    out = canciones[["track_id"]].copy().reset_index(drop=True)
    for p in personas:
        q = canciones.assign(persona=p, hora_local=momento)[["persona", "track_id", "artista", "album", "hora_local"]]
        prob = probabilidades(q, franja, tipo_dia)
        out[f"prob_{p}"] = prob
        out[f"pct_{p}"] = np.searchsorted(refs[p], prob, side="right") / max(len(refs[p]), 1)
    out["pct_grupo"] = out[[f"pct_{p}" for p in personas]].min(axis=1)
    out["entra"] = out["pct_grupo"] >= UMBRAL_PCT
    return out

personas = list(HISTORIALES)
conocidas = set(h.loc[h["hora_local"] < CORTE, "track_id"])
l2 = h[(h["hora_local"] >= CORTE) & ~h["track_id"].isin(conocidas)]
cand = l2.groupby("track_id")[["artista", "album"]].first().reset_index()
res = entraria(personas, cand, CORTE)
huecos = int(np.clip(round(N_LISTA * min(apertura(p, CORTE) for p in personas)), 2, 10))

# Resultado real en el lote 2 para cada miembro que escuchó la canción
real = prueba.set_index(["persona", "track_id"])["adopta"]
filas_ev = []
for p in personas:
    r = res.assign(adopta=real.reindex(pd.MultiIndex.from_arrays([[p] * len(res), res["track_id"]])).to_numpy())
    r = r[r["adopta"].notna()]
    top = r.sort_values("pct_grupo", ascending=False).head(huecos * 6)   # 6 viajes con la misma lista de huecos
    filas_ev.append({"persona": p, "canciones nuevas que escuchó": len(r),
                     "entran (n)": int(r["entra"].sum()),
                     "adopta · entran (%)": r.loc[r["entra"], "adopta"].mean() * 100,
                     "adopta · no entran (%)": r.loc[~r["entra"], "adopta"].mean() * 100,
                     "adopta · todas (%)": r["adopta"].mean() * 100})
evaluacion_o2 = pd.DataFrame(filas_ev).set_index("persona")
print(f"Candidatas (nuevas para los dos al empezar el lote 2): {len(res)} · entran: {int(res['entra'].sum())} "
      f"({res['entra'].mean():.1%}) · huecos por playlist de {N_LISTA}: {huecos}")
evaluacion_o2.round(1)

### 5.1 Ejemplo: una canción inventada

Una canción que **no existe** en ningún historial (`track_id` inventado). Se prueba con tres artistas:
- el artista más escuchado por los dos;
- un artista que solo conoce uno de ellos;
- un artista que nadie conoce.

Los nombres no se muestran.

In [ ]:
antes = h[h["hora_local"] < CORTE]
n_art = antes[antes["artista"] != "?"].pivot_table(index="artista", columns="persona", values="no_salto", aggfunc="size", fill_value=0)
ambos = n_art.min(axis=1).idxmax()
solo_uno = (n_art[personas[0]] * (n_art[personas[1]] == 0)).idxmax()
ejemplo = pd.DataFrame({"track_id": ["inventada_1", "inventada_2", "inventada_3"],
                        "artista": [ambos, solo_uno, "artista que nadie conoce"], "album": "?"})
r = entraria(personas, ejemplo, CORTE)
r.insert(1, "caso", ["artista muy escuchado por los dos", f"artista que solo conoce {personas[0]}", "artista desconocido"])
r.drop(columns="track_id").round(3)

## 6. Registro en MLflow y resultados guardados

In [ ]:
registrar("validacion novedades", {"ventana_adopcion_dias": VENTANA.days, "validacion": "2025-01..2025-03", "elegido": elegido},
          {"val_auc_" + re.sub(r"[^a-z0-9]+", "_", unicodedata.normalize("NFKD", i).encode("ascii", "ignore").decode().lower()).strip("_"): v
           for i, v in validacion["AUC"].items()}, [ruta_imp, ruta_ideas])
fila = lote2.loc[f"modelo ({elegido})"]
registrar("novedades · lote 2", {"modelo": nombre_el, "ventana": ventana_el, "variables": X_p.shape[1], "filas_entrenamiento": len(ent_fin),
                                 "umbral_pct": UMBRAL_PCT, "huecos": huecos},
          {"lote2_auc": fila["AUC"], "lote2_log_loss": fila["log loss"], "lote2_brier": fila["Brier"],
           "lote2_auc_artista_nuevo": fila["AUC artista nuevo"],
           "lote2_auc_ref_persona": lote2.iloc[0]["AUC"], "lote2_auc_ref_artista": lote2.iloc[1]["AUC"]} |
          {f"lote2_auc_{p}": fila[f"AUC {p}"] for p in personas} |
          {f"o2_adopta_entran_{p}": evaluacion_o2.loc[p, "adopta · entran (%)"] for p in personas}, [ruta_l2])

(RAIZ / "modelos").mkdir(exist_ok=True)
joblib.dump({"modelo": modelo, "columnas": list(X_p.columns), "ventana_adopcion_dias": VENTANA.days, "k": K,
             "dias_apertura": DIAS_APERTURA, "umbral_pct": UMBRAL_PCT, "elegido": elegido},
            RAIZ / "modelos" / "08_modelo_novedades.joblib")
json.dump({"elegido": elegido, "validacion": validacion.round(4).reset_index().to_dict("records"),
           "importancia": importancia.round(4).to_dict(), "lote2": lote2.round(4).reset_index().to_dict("records"),
           "utilidad_lote2": util.round(2).reset_index().to_dict("records"),
           "objetivo2": evaluacion_o2.round(2).reset_index().to_dict("records"), "huecos": huecos,
           "candidatas": len(res), "entran": int(res["entra"].sum())},
          open(SALIDA / "08_resumen.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1, default=float)
print("Guardado:", SALIDA / "08_resumen.json")

## 7. Conclusiones

*Cifras de la ejecución del 25/11/2026 con los historiales de ander y daniel.*

**Las dos ideas de partida se cumplen.**
- **La apertura a lo nuevo se puede medir y cambia con el tiempo.** Daniel pasó de un 24-28 % de escuchas de canciones nuevas (2020-21) a un 6 % (2026); Ander se mantiene en torno al 4-5 %. Por eso se mide en los últimos 180 días y no de una vez para siempre.
- **Conocer al artista sube la adopción.** Con un artista nuevo, Ander adopta el 34 % de las canciones y Daniel el 40 %; con un artista que ya ha escuchado 100 o más veces, el 43 % y el 54 %.
- **El shuffle afecta a saltar, no a que guste.** En la primera escucha en shuffle Ander no salta el 54 % y sin shuffle el 75 %, pero la adopción es casi igual (38 % y 38 %). Es otra razón para medir «adopta» y no «no salta».

**El modelo de novedades tiene una señal modesta pero real.**
- **Validación:** el GB con 18 meses gana por AUC (0,589 frente a 0,581 de la referencia de la persona). Su pérdida logarítmica es peor (se pasa de seguro), algo que se ve también en la calibración.
- **Lote 2, prueba única:** AUC de 0,637, frente a 0,630 de solo mirar la adopción con el artista y 0,585 de solo mirar a la persona. Casi toda la señal viene del artista; el modelo aporta sobre todo con Daniel (0,636 frente a 0,543).
- **Con un artista que la persona no conoce, el AUC es 0,568, casi azar:** sin datos de la canción en sí (género, sonido) no hay mucho que predecir.
- **En la práctica sí sirve para elegir.** Del 20 % de novedades que el modelo pone arriba, Ander adoptó el 30,5 % (frente al 18,8 % del resto) y Daniel el 48,1 % (frente al 33,2 %). Es una vez y media más.

**«¿Entraría?» en la playlist.**
- Con la regla del notebook 06 una canción nueva no entraba nunca.
- Con la regla de esta sección, de 7.010 canciones nuevas para los dos al empezar el lote 2 **entran 44 (0,6 %)**, en 3 huecos por playlist de 50.
- **Las que entran se adoptan más:** Daniel adoptó el 64,7 % de las que entran frente al 35,6 % de las que no. Ander, el 28,6 % frente al 21,8 %, aunque son solo 21 canciones y la cifra es poco segura.
- La regla es exigente a propósito: una canción tiene que estar en el 20 % mejor **para los dos**.

**Límites.**
- **Solo hay dos personas.** Las variables de apertura casi identifican a quién es, así que no se puede demostrar que funcionen igual con una persona nueva.
- **Calibración:** en el lote 2 la adopción real bajó (Ander del 31 % al 22 %) y el modelo sobreestima las probabilidades altas. Para decidir solo se usa el **orden** (percentiles), no la probabilidad en sí.
- **«Adoptar» no es exactamente disfrutarla en el viaje:** volver a escucharla también depende de que Spotify te la vuelva a poner.
- **Parámetros fijados antes de mirar el lote 2, sin ajustar:** el umbral de 0,80, la ventana de 90 días y la regla de huecos.

**Siguiente paso:**
- Llevarlo a la API: `POST /objetivo2/entraria` con el modelo `modelos/08_modelo_novedades.joblib` y huecos para novedades en `/objetivo2/playlist`.
- Probar variables de la canción en sí (género o sonido, por ejemplo del dataset de Kaggle) para el caso del artista desconocido.

## 8. Versiones de las librerías

In [ ]:
from importlib.metadata import version, PackageNotFoundError
for p in ["pandas", "numpy", "scikit-learn", "matplotlib", "mlflow", "joblib"]:
    try:
        print(f"{p}=={version(p)}")
    except PackageNotFoundError:
        print(f"# {p}: no instalado en este entorno")
print(f"# Python {platform.python_version()} · {'Google Colab' if EN_COLAB else 'local'} · {pd.Timestamp.now(tz=ZONA):%Y-%m-%d}")